# Phase 4 — Document Ingestion with Qdrant

**Pipeline:** PDF → page-aware extraction → chunks → embeddings → Qdrant → semantic search.

Topics: digital vs scanned PDFs, page/source metadata, heading/section heuristics, tables, chunk size/overlap, duplicate chunks, and incremental indexing.

Sample PDF: [Attention Is All You Need (arXiv)](https://arxiv.org/pdf/1706.03762). It has selectable text, section headings, and tables. Qdrant runs in local embedded mode here, so its collections will not appear at `http://localhost:6333/dashboard` unless you use a separate Qdrant server.

Run cells from top to bottom. First run downloads the PDF and embedding model.

## 1. Install dependencies
Run in a notebook cell; restart the kernel if prompted.

In [ ]:
%pip install -U qdrant-client sentence-transformers pypdf pdfplumber requests

## 2. Imports and configuration

In [2]:
from pathlib import Path
import hashlib, json, re, requests
from pypdf import PdfReader
import pdfplumber
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from uuid import uuid5, NAMESPACE_URL

PDF_URL = "https://arxiv.org/pdf/1706.03762"
DATA_DIR = Path("phase4_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATH = DATA_DIR / "attention_is_all_you_need.pdf"
DB_PATH = str((DATA_DIR / "qdrant_storage").resolve())
COLLECTION_NAME = "phase4_pdf_chunks"
DOCUMENT_ID = "attention_is_all_you_need"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE, CHUNK_OVERLAP = 900, 150

print("PDF1:", PDF_PATH.resolve())
print("Qdrant local storage:", DB_PATH)

PDF1: F:\MANIKANDAN-Office\RAG\notebooks\phase4_data\attention_is_all_you_need.pdf
Qdrant local storage: F:\MANIKANDAN-Office\RAG\notebooks\phase4_data\qdrant_storage


## 3. Download the sample PDF

In [3]:
if not PDF_PATH.exists():
    r = requests.get(PDF_URL, timeout=60)
    r.raise_for_status()
    if not r.content.startswith(b"%PDF"):
        raise ValueError("Download did not return a PDF")
    PDF_PATH.write_bytes(r.content)
print(f"{PDF_PATH.name}: {PDF_PATH.stat().st_size:,} bytes")

attention_is_all_you_need.pdf: 2,215,244 bytes


## 4. Extract text page by page

Keep each page separate so search results can cite the source page. `pypdf` extracts selectable text but does not OCR scanned images.

In [4]:
reader = PdfReader(str(PDF_PATH))
pages = [{"page_number": i, "text": (p.extract_text() or "").strip()}
         for i, p in enumerate(reader.pages, start=1)]
print("Pages:", len(pages))
for p in pages[:3]:
    print(f"\\n--- page {p['page_number']} ({len(p['text'])} chars) ---")
    print(p["text"][:400])

Pages: 15
\n--- page 1 (2859 chars) ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
G
\n--- page 2 (4257 chars) ---
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [ 35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-dec
\n--- page 3 (1826 chars) ---
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture 

## 5. Detect likely scanned/image pages

This is only a heuristic. Few extracted characters can mean a scanned page, a mostly graphical page, or extraction trouble.

In [5]:
LOW_TEXT_THRESHOLD = 40
page_diagnostics = [
    {"page_number": p["page_number"], "characters": len(p["text"]),
     "likely_needs_ocr": len(p["text"]) < LOW_TEXT_THRESHOLD}
    for p in pages
]
page_diagnostics[:10]

[{'page_number': 1, 'characters': 2859, 'likely_needs_ocr': False},
 {'page_number': 2, 'characters': 4257, 'likely_needs_ocr': False},
 {'page_number': 3, 'characters': 1826, 'likely_needs_ocr': False},
 {'page_number': 4, 'characters': 2503, 'likely_needs_ocr': False},
 {'page_number': 5, 'characters': 3185, 'likely_needs_ocr': False},
 {'page_number': 6, 'characters': 3477, 'likely_needs_ocr': False},
 {'page_number': 7, 'characters': 3321, 'likely_needs_ocr': False},
 {'page_number': 8, 'characters': 3193, 'likely_needs_ocr': False},
 {'page_number': 9, 'characters': 2971, 'likely_needs_ocr': False},
 {'page_number': 10, 'characters': 3112, 'likely_needs_ocr': False}]

### Optional OCR for scanned PDFs

For OCR, install Tesseract on your operating system and then `%pip install pymupdf pytesseract pillow`. Uncomment and run this example only when needed. OCR text can contain errors, so inspect it before indexing.

In [6]:
# OPTIONAL — requires Tesseract installed on your OS
# import fitz, pytesseract, io
# from PIL import Image
# def ocr_page(pdf_path, zero_based_page_index, dpi=200):
#     doc = fitz.open(pdf_path)
#     pix = doc[zero_based_page_index].get_pixmap(dpi=dpi)
#     return pytesseract.image_to_string(Image.open(io.BytesIO(pix.tobytes("png"))))
# print(ocr_page(PDF_PATH, 0)[:1000])

## 6. Detect headings and inspect tables

Heading detection here is deliberately heuristic. PDF extraction often loses layout, so do not treat these labels as perfect document structure.

In [7]:
def looks_like_heading(line):
    line = line.strip()
    if not line or len(line) > 100:
        return False
    if re.match(r"^\\d+(?:\\.\\d+)*\\s+\\S+", line):
        return True
    return bool(re.match(r"^[A-Z][A-Za-z0-9 ,:;()’'/-]{2,90}$", line)
                and len(line.split()) <= 12)

for p in pages:
    headings = [x.strip() for x in p["text"].splitlines() if looks_like_heading(x)]
    if headings:
        print("Page", p["page_number"], headings[:5])

Page 1 ['Provided proper attribution is provided, Google hereby grants permission to', 'Attention Is All You Need', 'Google Brain', 'Google Brain', 'Google Research']
Page 4 ['Scaled Dot-Product Attention', 'Multi-Head Attention']
Page 5 ['The Transformer uses multi-head attention in three different ways:']
Page 6 ['Layer Type Complexity per Layer Sequential Maximum Path Length', 'Operations', 'In this work, we use sine and cosine functions of different frequencies:']
Page 7 ['We employ three types of regularization during training:']
Page 8 ['Model', 'BLEU Training Cost (FLOPs)', 'EN-DE EN-FR EN-DE EN-FR']
Page 10 ['Parser Training WSJ 23 F1', 'References']
Page 13 ['Attention Visualizations', 'Input-Input Layer5', 'American', 'American']
Page 14 ['Input-Input Layer5', 'The', 'Law', 'The', 'Law']
Page 15 ['Input-Input Layer5', 'The', 'Law', 'The', 'Law']


In [8]:
# Inspect table extraction using pdfplumber. Results vary with PDF layout.
table_records = []
with pdfplumber.open(str(PDF_PATH)) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        for table_index, table in enumerate(page.extract_tables(), start=1):
            table_records.append({"page_number": page_number,
                                  "table_index": table_index, "rows": table})
print("Detected tables:", len(table_records))
table_records[:1]

Detected tables: 10


[{'page_number': 9,
  'table_index': 1,
  'rows': [['train\nN d d h d d P ϵ\nmodel ff k v drop ls steps'],
   ['6 512 2048 8 64 64 0.1 0.1 100K'],
   ['1 512 512\n4 128 128\n16 32 32\n32 16 16'],
   ['16\n32'],
   ['2\n4\n8\n256 32 32\n1024 128 128\n1024\n4096'],
   ['0.0\n0.2\n0.0\n0.2'],
   ['positionalembeddinginsteadofsinusoids'],
   ['6 1024 4096 16 0.3 300K']]}]

### Table handling

Tables may extract as rows, plain text, or imperfect fragments. For retrieval, keep column labels with their values and add table/page metadata. Example serializer:

In [9]:
def table_to_text(rows, table_name="Extracted table"):
    lines = []
    for i, row in enumerate(rows, start=1):
        values = [(cell or "").strip().replace("\\n", " ") for cell in row]
        if any(values):
            lines.append(f"{table_name} | row {i}: " + " | ".join(values))
    return "\\n".join(lines)

if table_records:
    print(table_to_text(table_records[0]["rows"][:5], "PDF table preview"))
else:
    print("No tables detected by this extraction pass.")

PDF table preview | row 1: train
N d d h d d P ϵ
model ff k v drop ls steps\nPDF table preview | row 2: 6 512 2048 8 64 64 0.1 0.1 100K\nPDF table preview | row 3: 1 512 512
4 128 128
16 32 32
32 16 16\nPDF table preview | row 4: 16
32\nPDF table preview | row 5: 2
4
8
256 32 32
1024 128 128
1024
4096


## 7. Attach section context to each page
The current heading is a best-effort heuristic; page number is the more reliable provenance.

In [10]:
page_records = []
current_section = "Unknown section"
for p in pages:
    if not p["text"]:
        continue
    for line in p["text"].splitlines():
        if looks_like_heading(line):
            current_section = line.strip()
    page_records.append({
        "page_number": p["page_number"], "text": p["text"],
        "section": current_section, "source_url": PDF_URL,
        "source_file": PDF_PATH.name
    })
print("Pages with text:", len(page_records))

Pages with text: 15


## 8. Chunk text with overlap
Character chunking is easy to learn, but can split sentences or tables. Later compare with paragraph-, sentence-, or token-based chunking.

In [11]:
def split_text(text, chunk_size=900, overlap=150):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Require chunk_size > 0 and 0 <= overlap < chunk_size")
    text = re.sub(r"\\s+", " ", text).strip()
    out, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start + int(chunk_size * 0.65):
                end = boundary
        piece = text[start:end].strip()
        if piece:
            out.append(piece)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return out

chunks = []
for p in page_records:
    for idx, text in enumerate(split_text(p["text"], CHUNK_SIZE, CHUNK_OVERLAP)):
        chunks.append({
            "text": text, "page_number": p["page_number"], "section": p["section"],
            "source_url": p["source_url"], "source_file": p["source_file"],
            "chunk_index_on_page": idx
        })
print("Chunks:", len(chunks))
print(chunks[0] if chunks else "No chunks")

Chunks: 59
{'text': 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the', 'page_number': 1, 'section': 'The dominant sequence tr

## 9. Chunk-size experiment

In [12]:
lengths = [len(c["text"]) for c in chunks]
if lengths:
    print("min/max/average:", min(lengths), max(lengths), round(sum(lengths)/len(lengths), 1))
for size, overlap in [(500, 80), (900, 150), (1400, 200)]:
    count = sum(len(split_text(p["text"], size, overlap)) for p in page_records)
    print(f"chunk_size={size}, overlap={overlap} -> {count} chunks")

min/max/average: 177 899 782.8
chunk_size=500, overlap=80 -> 99 chunks
chunk_size=900, overlap=150 -> 59 chunks
chunk_size=1400, overlap=200 -> 38 chunks


## 10. Remove exact duplicate chunks

A SHA-256 hash of normalized text acts as a content fingerprint. Here we keep the first exact occurrence. Depending on your use case, repeated passages from different pages may need to be retained instead.

In [13]:
def normalize(text):
    return re.sub(r"\\s+", " ", text).strip()

def content_hash(text):
    return hashlib.sha256(normalize(text).encode("utf-8")).hexdigest()

unique_chunks, seen = [], set()
for chunk in chunks:
    digest = content_hash(chunk["text"])
    if digest in seen:
        continue
    seen.add(digest)
    item = dict(chunk)
    item["content_hash"] = digest
    unique_chunks.append(item)

print("Original:", len(chunks), "Unique:", len(unique_chunks),
      "Duplicates skipped:", len(chunks) - len(unique_chunks))

Original: 59 Unique: 59 Duplicates skipped: 0


## 11. Create embeddings

In [14]:
model = SentenceTransformer(MODEL_NAME)
VECTOR_SIZE = model.get_sentence_embedding_dimension()
print("Model:", MODEL_NAME, "| dimensions:", VECTOR_SIZE)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model: sentence-transformers/all-MiniLM-L6-v2 | dimensions: 384


C:\Users\manit\AppData\Local\Temp\ipykernel_5128\340623442.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  VECTOR_SIZE = model.get_sentence_embedding_dimension()


## 12. Connect to local Qdrant

Local embedded mode stores data in `DB_PATH`. Do not open the same local path from multiple Python processes at once. To use the web dashboard, run a Qdrant server separately and use `QdrantClient(url="http://localhost:6333")` instead.

In [15]:
client = QdrantClient(url="http://localhost:6333")
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
    )
print(client.get_collections())

collections=[CollectionDescription(name='phase3_learning'), CollectionDescription(name='phase3_learning1'), CollectionDescription(name='phase4_pdf_chunks')]


## 13. Incremental indexing with a manifest

The manifest records the original PDF checksum. If unchanged, skip re-indexing. If changed, delete old points for this document and add new chunks. A production pipeline should also fingerprint chunking settings, parser version, and embedding model.

In [16]:
MANIFEST_PATH = DATA_DIR / "ingestion_manifest.json"

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def load_manifest():
    return json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else {}

def save_manifest(data):
    MANIFEST_PATH.write_text(json.dumps(data, indent=2), encoding="utf-8")

current_checksum = file_sha256(PDF_PATH)
manifest = load_manifest()
previous_checksum = manifest.get(DOCUMENT_ID, {}).get("checksum")
print("Current checksum:", current_checksum)
print("Previous checksum:", previous_checksum)
print("Changed since last indexing:", previous_checksum is not None and previous_checksum != current_checksum)

Current checksum: bdfaa68d8984f0dc02beaca527b76f207d99b666d31d1da728ee0728182df697
Previous checksum: None
Changed since last indexing: False


## 14. Embed and upsert only when needed
Point IDs are deterministic UUIDs based on document ID + content hash. The payload stores text and provenance fields.

In [ ]:
def point_id(document_id, digest):
    return str(uuid5(NAMESPACE_URL, f"{document_id}:{digest}"))

manifest = load_manifest()
previous_checksum = manifest.get(DOCUMENT_ID, {}).get("checksum")

if previous_checksum == current_checksum:
    print("PDF checksum unchanged; skipping indexing.")
else:
    if previous_checksum is not None:
        client.delete(
            collection_name=COLLECTION_NAME,
            points_selector=models.FilterSelector(
                filter=models.Filter(must=[
                    models.FieldCondition(key="document_id",
                                          match=models.MatchValue(value=DOCUMENT_ID))
                ])
            ),
            wait=True,
        )
    if unique_chunks:
        vectors = model.encode([c["text"] for c in unique_chunks],
                               normalize_embeddings=True, show_progress_bar=True).tolist()
        points = []
        for c, vector in zip(unique_chunks, vectors):
            payload = {
                "document_id": DOCUMENT_ID, "text": c["text"],
                "page_number": c["page_number"], "section": c["section"],
                "source_url": c["source_url"], "source_file": c["source_file"],
                "chunk_index_on_page": c["chunk_index_on_page"],
                "content_hash": c["content_hash"], "document_checksum": current_checksum
            }
            points.append(models.PointStruct(id=point_id(DOCUMENT_ID, c["content_hash"]),
                                             vector=vector, payload=payload))
        client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
    manifest[DOCUMENT_ID] = {"checksum": current_checksum, "source_url": PDF_URL,
                             "source_file": PDF_PATH.name, "chunk_count": len(unique_chunks)}
    save_manifest(manifest)
    print("Indexed", len(unique_chunks), "unique chunks.")

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

## 15. Verify stored points

In [ ]:
print(client.get_collection(COLLECTION_NAME))
points, next_page = client.scroll(collection_name=COLLECTION_NAME, limit=3,
                                  with_payload=True, with_vectors=False)
for p in points:
    print({"page": p.payload.get("page_number"), "section": p.payload.get("section"),
           "source": p.payload.get("source_file"), "preview": p.payload.get("text", "")[:180]})

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=59 segments_count=6 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, preven

## 16. Semantic search with page/source metadata

In [ ]:
def search_pdf(query, top_k=5):
    query_vector = model.encode(query, normalize_embeddings=True).tolist()
    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=models.Filter(must=[
            models.FieldCondition(key="document_id",
                                  match=models.MatchValue(value=DOCUMENT_ID))
        ]),
        limit=top_k, with_payload=True
    )
    return [{
        "score": round(float(p.score), 4),
        "page": p.payload.get("page_number"),
        "section": p.payload.get("section"),
        "source": p.payload.get("source_url"),
        "text": p.payload.get("text", "")
    } for p in response.points]

results = search_pdf("Why does the Transformer use attention instead of recurrence?", 3)
for i, r in enumerate(results, 1):
    print(f"\\n{i}. score={r['score']} | page={r['page']} | section={r['section']}")
    print(r["text"][:650])
    print("Source:", r["source"])

\n1. score=0.5758 | page=5 | section=The Transformer uses multi-head attention in three different ways:
gle-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].
• The encoder contains self-attention layers. In a self-attention layer all of the keys, values
and queries come from the same p
Source: https://arxiv.org/pdf/1706.03762
\n2. score=0.5417 | page=2 | section=The dominant sequence transduction models are based on complex recurrent or
ure precludes parallelization within training examples, which becomes critical at longer
se

## 17. Metadata filtering example

In [ ]:
qvec = model.encode("What are the main contributions?", normalize_embeddings=True).tolist()
response = client.query_points(
    collection_name=COLLECTION_NAME, query=qvec, limit=3, with_payload=True,
    query_filter=models.Filter(must=[
        models.FieldCondition(key="document_id", match=models.MatchValue(value=DOCUMENT_ID)),
        models.FieldCondition(key="page_number", range=models.Range(gte=1, lte=5))
    ])
)
for p in response.points:
    print("score:", round(float(p.score), 4), "page:", p.payload.get("page_number"))
    print(p.payload.get("text", "")[:350], "\\n")

score: 0.1993 page: 3
sition, ensures that the
predictions for position i can depend only on the known outputs at positions less than i.
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3 \n
score: 0.1971 page: 4
hanisms perform similarly, additive attention outperforms
dot product attention without scaling for larger values of dk [3]. We suspect that for large values of
dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has
extremely small gradients 4. To counteract this effect, we scale the dot products by 1√d \n
score: 0.1821 page: 4
together
into a matrix Q. The keys and values are also packed together into matrices K and V . We compute
the matrix of outputs as:
Attention(Q, K, V) = softmax(QK T√dk )V (1)
The two most commonly used attention functions are additive attention

## 18. Incremental indexing improvements

The current checksum detects source PDF changes, but not changes to chunk size, overlap, parser, or embedding model. A stronger pipeline fingerprint should combine all of these settings. If the fingerprint changes, re-index.

Try modifying the downloaded PDF, or point `PDF_URL` to another PDF and assign a different `DOCUMENT_ID`. For multiple documents, keep the `document_id` filter in searches so results do not mix accidentally.

In [ ]:
def pipeline_fingerprint(file_checksum, chunk_size, overlap, model_name):
    config = {"file_checksum": file_checksum, "chunk_size": chunk_size,
              "overlap": overlap, "embedding_model": model_name}
    return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()

print("Pipeline fingerprint:",
      pipeline_fingerprint(current_checksum, CHUNK_SIZE, CHUNK_OVERLAP, MODEL_NAME))

Pipeline fingerprint: c7ba22ecb7973c9f30084004e536b9e8085ca41622738a849b1b5a95ccc60fee


## 19. Exercises and takeaways

1. Index another PDF using a new `DOCUMENT_ID`.
2. Compare chunk sizes 500, 900, and 1400 characters.
3. Test five questions and note whether the right page appears in the top 3.
4. Try a scanned PDF and compare extraction before/after OCR.
5. Compare extracted table rows with plain text extraction.
6. Add the pipeline fingerprint to the manifest so changing chunking/model settings triggers re-indexing.

**Takeaways:** preserve page/source metadata; OCR scanned pages; inspect headings/tables; deduplicate intentionally; and make ingestion repeatable with checksums. `client.close()` closes the local database handle; it does not delete its files.

In [ ]:
client.close()\nprint('Closed local Qdrant client.')